# Credit Card Default (Fair DP-GANs) 

Author: Ilse Harmers \
Last modified: February 24, 2026

In [ ]:
# Importing libraries.
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
from snsynth import Synthesizer
import snsynth.transform as tf
import utils
import warnings
warnings.filterwarnings("ignore", message = r"Using", category = FutureWarning)

In [ ]:
# Importing train set.
credit_train = pd.read_csv("./train-test-datasets/credit-card-default/credit_train.csv")

# Preprocessing the dataset for the Fair DP-GANs; the dataset Fair DP-GANs are expecting the first two columns in the original dataset to be the 
# sensitive and target attributes. 
cols = credit_train.columns.to_list()
cols.remove("SEX")
cols.remove("DEFAULT")
credit_train = credit_train[["SEX", "DEFAULT"] + cols]

print(credit_train.columns.to_list())
credit_train.describe()

In [ ]:
# Setting up preprocessor table transformer.

tt = tf.TableTransformer([
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # SEX
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # DEFAULT
    tf.ChainTransformer([
        tf.LogTransformer(),
        tf.MinMaxTransformer(lower = np.log(credit_train["LIMIT_BAL"].min()), 
                             upper = np.log(credit_train["LIMIT_BAL"].max()),
                             negative = False) # LIMIT_BAL; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # EDUCATION
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # MARRIAGE
    tf.MinMaxTransformer(lower = credit_train["AGE"].min(), 
                         upper = credit_train["AGE"].max(),
                         negative = False), # AGE; scaling to range (0, 1)
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_0
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_2
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_3
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_4
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_5
    tf.ChainTransformer([tf.LabelTransformer(), tf.OneHotEncoder()]), # PAY_6
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-166000.0) + 1)), 
                             upper = np.log(965000.0 + 1), negative = False) # BILL_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-70000.0) + 1)), 
                             upper = np.log(984000.0 + 1), negative = False) # BILL_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-157000.0) + 1)), 
                             upper = np.log(1664000.0 + 1), negative = False) # BILL_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-170000.0) + 1)), 
                             upper = np.log(892000.0 + 1), negative = False) # BILL_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-81000.0) + 1)), 
                             upper = np.log(927000.0 + 1), negative = False) # BILL_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = -1 * (np.log(abs(-340000.0) + 1)), 
                             upper = np.log(962000.0 + 1), negative = False) # BILL_AMT6; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(874000.0 + 1), 
                             negative = False) # PAY_AMT1; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(1684000.0 + 1), 
                             negative = False) # PAY_AMT2; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(896000.0 + 1), 
                             negative = False) # PAY_AMT3; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(621000.0 + 1), 
                             negative = False) # PAY_AMT4; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(427000.0 + 1), 
                             negative = False) # PAY_AMT5; scaling to range (0, 1)
    ]),
    tf.ChainTransformer([
        tf.LogModulusTransformer(),
        tf.MinMaxTransformer(lower = 0, upper = np.log(529000.0 + 1), 
                             negative = False) # PAY_AMT6; scaling to range (0, 1)
    ]),
])

In [ ]:
# Defining delta as the inverse of the size of the dataset: if a dataset has 9 * 10^3 rows, then delta = 10^(-4). 
delta = 10**np.floor(np.log10(1/credit_train.shape[0]))
print(delta)

# Defining beta1 in Adam optimizer.
beta1 = 0.9 

# Defining epsilon value.
epsi = 1

# Defining fairness loss: [dem, dis, dem-dis].
fair_loss = "dem-dis"

In [ ]:
r = 1
while r < 6:
    
    print(f"Run {r}")

    # Synthesizing the dataset with one of the Fair DP-GANs.
    synth = Synthesizer.create('fairdpgan', s = "SEX", s_unpriv = 2, s_priv = 1, y = "DEFAULT", y_des = 1, fair_loss = fair_loss,
                               epsilon = epsi, delta = delta, beta1 = beta1, plot_losses = True, batch_size = 256, dataset = "Credit",
                file_path = f"./Experiment_SensitiveBalancingPre/synthetic-datasets_FairDP-GAN({fair_loss})/credit-card-default/epsi_{epsi}/run{r}/",
                               #sanity_check = True
                              )
    synth.fit(credit_train, transformer = tt, preprocessor_eps = 0.0)

    try:
        # Generating the first synthetic dataset.
        sample1 = synth.sample(credit_train.shape[0])
        # Encoding the sensitive attribute for the fairness analysis. Note regarding 'SEX': 1 = male; 2 = female.
        sample1_encoded = pd.concat([utils.one_hot_encode(sample1[["SEX"]], order = [[2, 1]]).reset_index(drop = True),
                                    sample1["DEFAULT"].reset_index(drop = True)], axis = 1)
        dem1 = utils.demographic_parity(df = sample1_encoded, s = "SEX", y = "DEFAULT")
        dis1 = utils.disparate_impact(df = sample1_encoded, s = "SEX", y = "DEFAULT")
        print("Sample 1 [dem., dis.]: ",  [dem1, dis1])
        
        # Generating the second synthetic dataset.
        sample2 = synth.sample(credit_train.shape[0])
        # Encoding the sensitive attribute for the fairness analysis. Note regarding 'SEX': 1 = male; 2 = female.
        sample2_encoded = pd.concat([utils.one_hot_encode(sample2[["SEX"]], order = [[2, 1]]).reset_index(drop = True),
                                    sample2["DEFAULT"].reset_index(drop = True)], axis = 1)
        dem2 = utils.demographic_parity(df = sample2_encoded, s = "SEX", y = "DEFAULT")
        dis2 = utils.disparate_impact(df = sample2_encoded, s = "SEX", y = "DEFAULT")
        print("Sample 2 [dem., dis.]: ",  [dem2, dis2])

        # Generating the third synthetic dataset.
        sample3 = synth.sample(credit_train.shape[0])
        # Encoding the sensitive attribute for the fairness analysis. Note regarding 'SEX': 1 = male; 2 = female.
        sample3_encoded = pd.concat([utils.one_hot_encode(sample3[["SEX"]], order = [[2, 1]]).reset_index(drop = True),
                                    sample3["DEFAULT"].reset_index(drop = True)], axis = 1)
        dem3 = utils.demographic_parity(df = sample3_encoded, s = "SEX", y = "DEFAULT")
        dis3 = utils.disparate_impact(df = sample3_encoded, s = "SEX", y = "DEFAULT")
        print("Sample 3 [dem., dis.]: ",  [dem3, dis3])

        # Saving the synthetic datasets.
        sample1.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})/credit-card-default/epsi_{epsi}/run{r}/sample1.csv", index = False)
        sample2.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})/credit-card-default/epsi_{epsi}/run{r}/sample2.csv", index = False)
        sample3.to_csv(f"./synthetic-datasets_FairDP-GAN({fair_loss})/credit-card-default/epsi_{epsi}/run{r}/sample3.csv", index = False)
        
        r += 1
    except ZeroDivisionError:
        r += 0